# Advanced Problems with Solutions: Object Mutability

This notebook explores **object mutability** in Python, with a focus on **lists**, **dictionaries**, **tuples**, **aliasing**, **shallow vs deep copying**, and the distinction between **mutating an object** and **re-binding a variable name**.

## Learning goals
- Understand what it means for an object to be **mutable** or **immutable**.
- Use `id()` to distinguish **in-place mutation** from **re-assignment**.
- Predict how aliasing affects program behavior.
- Recognize subtle cases such as **immutable containers holding mutable objects**.
- Apply best practices to avoid bugs caused by unintended shared state.

> **Important note:** The exact values returned by `id()` vary across runs and Python implementations. The important question is whether identities stay the **same** or become **different**.


## Quick refresher

An object is **mutable** if its internal state can be changed without creating a new object.

Examples of common mutable built-in types:
- `list`
- `dict`
- `set`

Examples of common immutable built-in types:
- `int`
- `float`
- `str`
- `tuple`

Be careful: an **immutable container** may still contain **mutable objects**.


## Warm-up: mutating a list in place


In [1]:
my_list = [1, 2, 3]
print(my_list)
print(hex(id(my_list)))

my_list.append(4)
print(my_list)
print(hex(id(my_list)))


[1, 2, 3]
0x1c7a66ce4c0
[1, 2, 3, 4]
0x1c7a66ce4c0


The contents changed, but the identity stayed the same. That means the list was **mutated in place**.

Now compare that with re-binding:

In [2]:
my_list_1 = [1, 2, 3]
print(my_list_1)
print(hex(id(my_list_1)))

my_list_1 = my_list_1 + [4]
print(my_list_1)
print(hex(id(my_list_1)))


[1, 2, 3]
0x1c7a66ce240
[1, 2, 3, 4]
0x1c7a66ce380


Here the identity changes, so Python created a **new list object** and then re-bound `my_list_1` to that new object.

The same mutability idea applies to dictionaries:

In [3]:
my_dict = dict(key1='value 1')
print(my_dict)
print(hex(id(my_dict)))

my_dict['key1'] = 'modified value 1'
print(my_dict)
print(hex(id(my_dict)))

my_dict['key2'] = 'value 2'
print(my_dict)
print(hex(id(my_dict)))


{'key1': 'value 1'}
0x1c7a66e70c0
{'key1': 'modified value 1'}
0x1c7a66e70c0
{'key1': 'modified value 1', 'key2': 'value 2'}
0x1c7a66e70c0


The dictionary changes in place, so its identity stays the same.

Now recall a subtle point about tuples:

In [4]:
a = [1, 2]
b = [3, 4]
t = (a, b)
print('Before:', t)

a.append(3)
b.append(5)
print('After:', t)


Before: ([1, 2], [3, 4])
After: ([1, 2, 3], [3, 4, 5])


The tuple `t` is immutable, but the lists inside it are mutable. So the tuple still points to the same two list objects, but those list objects can change.


---
# Problem 1 - Mutation or re-assignment?

For each operation below, decide whether it **mutates the existing object** or **creates a new object and rebinds the variable**.


In [5]:
x = [10, 20, 30]
print('Initial:', x, '| id =', id(x))

x.append(40)
print('After append:', x, '| id =', id(x))

x = x + [50]
print('After concatenation:', x, '| id =', id(x))

x.extend([60])
print('After extend:', x, '| id =', id(x))


Initial: [10, 20, 30] | id = 1957002145600
After append: [10, 20, 30, 40] | id = 1957002145600
After concatenation: [10, 20, 30, 40, 50] | id = 1957002293312
After extend: [10, 20, 30, 40, 50, 60] | id = 1957002293312


## Solution 1

- `x.append(40)` **mutates** the existing list in place.
- `x = x + [50]` creates a **new list** and rebinds `x`.
- `x.extend([60])` **mutates** the current list in place.

**Best practice:** Learn which methods mutate in place (`append`, `extend`, `sort`, `update`) and which operators often create new objects (`+` on lists, many string operations).


---
# Problem 2 - Aliasing with mutable objects

Predict the output before running the code. Then explain why.


In [6]:
data = [1, 2, 3]
alias = data

alias.append(4)

print('data =', data)
print('alias =', alias)
print('data is alias ->', data is alias)


data = [1, 2, 3, 4]
alias = [1, 2, 3, 4]
data is alias -> True


## Solution 2

`alias = data` does not copy the list. It creates a second reference to the same object.

So when `alias.append(4)` mutates the list, both names reflect the change:

- `data == [1, 2, 3, 4]`
- `alias == [1, 2, 3, 4]`
- `data is alias` is `True`

**Key idea:** With mutable objects, aliasing means changes through one name are visible through all names pointing to that object.


---
# Problem 3 - Dictionary mutation vs re-binding

Determine which statements mutate the dictionary and which create a new object.


In [7]:
d = {'a': 1, 'b': 2}
print('Initial:', d, '| id =', id(d))

d['c'] = 3
print('After item assignment:', d, '| id =', id(d))

d.update({'d': 4})
print('After update:', d, '| id =', id(d))

d = {**d, 'e': 5}
print('After dictionary unpacking into new dict:', d, '| id =', id(d))


Initial: {'a': 1, 'b': 2} | id = 1957002376128
After item assignment: {'a': 1, 'b': 2, 'c': 3} | id = 1957002376128
After update: {'a': 1, 'b': 2, 'c': 3, 'd': 4} | id = 1957002376128
After dictionary unpacking into new dict: {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5} | id = 1957002367872


## Solution 3

- `d['c'] = 3` mutates the existing dictionary.
- `d.update({'d': 4})` also mutates the existing dictionary.
- `d = {**d, 'e': 5}` creates a **new dictionary** and rebinds `d`.

**Best practice:** When debugging dictionaries, track whether a line changes the current object or replaces it with a new one.


---
# Problem 4 - Immutable tuple, mutable contents

Explain why the following code works even though tuples are immutable.


In [8]:
left = [1, 2]
right = [3, 4]
t = (left, right)

print('Before:', t, '| id(t) =', id(t))
left.append(99)
right[0] = -3
print('After:', t, '| id(t) =', id(t))


Before: ([1, 2], [3, 4]) | id(t) = 1957002104640
After: ([1, 2, 99], [-3, 4]) | id(t) = 1957002104640


## Solution 4

The tuple `t` is immutable, which means the tuple cannot be changed to point to different elements after creation.

However, the tuple's elements are two **lists**, and lists are mutable. So the lists themselves can be modified.

- The identity of `t` stays the same.
- The tuple still contains references to the same two list objects.
- The contents of those list objects change.

**Subtle but important:** The tuple did not change its elements. The objects referenced by those elements changed internally.


---
# Problem 5 - Shallow copy trap

This problem tests whether you can distinguish copying the outer container from copying nested objects.


In [9]:
original = [[1, 2], [3, 4]]
copy1 = original.copy()

copy1[0].append(99)
copy1.append(['new'])

print('original =', original)
print('copy1 =', copy1)
print('original is copy1 ->', original is copy1)
print('original[0] is copy1[0] ->', original[0] is copy1[0])


original = [[1, 2, 99], [3, 4]]
copy1 = [[1, 2, 99], [3, 4], ['new']]
original is copy1 -> False
original[0] is copy1[0] -> True


## Solution 5

`original.copy()` makes a **shallow copy** of the outer list.

So:
- `original` and `copy1` are different outer lists.
- But their inner lists are still shared.

Therefore:
- `copy1[0].append(99)` affects both `original` and `copy1`.
- `copy1.append(['new'])` affects only `copy1`, because that changes the outer list structure.

Typical result:
- `original == [[1, 2, 99], [3, 4]]`
- `copy1 == [[1, 2, 99], [3, 4], ['new']]`

**Best practice:** Use deep copying when you need independent nested mutable objects.


---
# Problem 6 - Deep copy for independence

Use `copy.deepcopy` to avoid the bug from the previous problem.


In [10]:
import copy

original = [[1, 2], [3, 4]]
copy2 = copy.deepcopy(original)

copy2[0].append(99)
copy2.append(['new'])

print('original =', original)
print('copy2 =', copy2)
print('original is copy2 ->', original is copy2)
print('original[0] is copy2[0] ->', original[0] is copy2[0])


original = [[1, 2], [3, 4]]
copy2 = [[1, 2, 99], [3, 4], ['new']]
original is copy2 -> False
original[0] is copy2[0] -> False


## Solution 6

`copy.deepcopy(original)` recursively copies nested objects.

That means both the outer list and the inner lists are independent in `copy2`.

So changes to `copy2` do not affect `original`.

**Rule of thumb:**
- Use a shallow copy when shared nested objects are acceptable.
- Use a deep copy when full structural independence is required.


---
# Problem 7 - Function side effects

Predict what the caller will see after each function call.


In [11]:
def mutate_list(values):
    values.append(100)

def rebind_list(values):
    values = values + [200]

nums1 = [1, 2, 3]
nums2 = [1, 2, 3]

mutate_list(nums1)
rebind_list(nums2)

print('nums1 =', nums1)
print('nums2 =', nums2)


nums1 = [1, 2, 3, 100]
nums2 = [1, 2, 3]


## Solution 7

- `mutate_list(nums1)` changes the original list, so `nums1` becomes `[1, 2, 3, 100]`.
- `rebind_list(nums2)` only changes the local parameter name inside the function, so `nums2` remains `[1, 2, 3]`.

**Best practice:** Make mutation explicit in function names, documentation, or type hints when possible. Hidden side effects make code harder to reason about.


---
# Problem 8 - Mutable default argument bug

This is one of the most famous Python mutability pitfalls. Identify the bug and fix it.


In [12]:
def add_item(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_item('a'))
print(add_item('b'))
print(add_item('c'))


['a']
['a', 'b']
['a', 'b', 'c']


## Solution 8

The default list `bucket=[]` is created once, at function definition time, not each time the function is called.

Because lists are mutable, all calls without an explicit `bucket` argument share the same list.

A correct version is:


In [13]:
def add_item_fixed(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(add_item_fixed('a'))
print(add_item_fixed('b'))
print(add_item_fixed('c'))


['a']
['b']
['c']


**Best practice:** Never use a mutable object as a default argument unless you intentionally want shared state across calls.


---
# Problem 9 - Nested repetition bug

Explain why the following matrix construction is wrong, and then fix it.


In [14]:
grid = [[0] * 3] * 4
grid[0][1] = 7
print(grid)


[[0, 7, 0], [0, 7, 0], [0, 7, 0], [0, 7, 0]]


## Solution 9

The expression `[[0] * 3] * 4` creates one inner list and repeats references to that same list four times.

So mutating one row mutates them all.

A correct construction is:


In [15]:
grid = [[0] * 3 for _ in range(4)]
grid[0][1] = 7
print(grid)


[[0, 7, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]


This comprehension creates a new inner list on each iteration, so each row is independent.


---
# Problem 10 - Advanced tracing challenge

Trace all objects carefully and determine the final values and aliasing relationships.


In [16]:
a = {'x': [1, 2], 'y': [3, 4]}
b = a
c = a.copy()

a['x'].append(99)
b['z'] = ['new']
c['y'] = ['replaced']
a = {'fresh': True}

print('a =', a, '| id(a) =', id(a))
print('b =', b, '| id(b) =', id(b))
print('c =', c, '| id(c) =', id(c))
print("b['x'] is c['x'] ->", b['x'] is c['x'])
print("'z' in c ->", 'z' in c)


a = {'fresh': True} | id(a) = 1957002135360
b = {'x': [1, 2, 99], 'y': [3, 4], 'z': ['new']} | id(b) = 1957002134720
c = {'x': [1, 2, 99], 'y': ['replaced']} | id(c) = 1957002148992
b['x'] is c['x'] -> True
'z' in c -> False


## Solution 10

Step by step:

1. `a` is a dictionary whose values are two lists.
2. `b = a` makes `b` an alias of the same dictionary.
3. `c = a.copy()` creates a shallow copy of the dictionary.
   - The outer dictionary is new.
   - The lists under `'x'` and `'y'` are still shared initially.
4. `a['x'].append(99)` mutates the shared list stored under `'x'`, so both `b['x']` and `c['x']` reflect the change.
5. `b['z'] = ['new']` mutates the original dictionary shared by `a` and `b`, but not `c`.
6. `c['y'] = ['replaced']` changes only `c`'s outer dictionary binding for key `'y'`.
7. `a = {'fresh': True}` rebinds `a` to a completely new dictionary, leaving `b` unchanged.

Final state:
- `a == {'fresh': True}`
- `b == {'x': [1, 2, 99], 'y': [3, 4], 'z': ['new']}`
- `c == {'x': [1, 2, 99], 'y': ['replaced']}`
- `b['x'] is c['x']` is `True`
- `'z' in c` is `False`

**Master takeaway:** Shallow copies duplicate the outer container, but nested mutable objects may still be shared.


---
# Summary

## Core rules
- Mutable objects can change their contents without changing identity.
- Immutable objects cannot change in place.
- Re-binding a variable changes what name points to; it does not mutate the old object.
- Aliasing means multiple names refer to the same object.
- Shallow copies duplicate only one container level.
- Deep copies recursively copy nested objects.

## Best practices
- Be explicit about whether code mutates inputs.
- Avoid mutable default arguments.
- Be cautious with nested mutable structures.
- Prefer comprehensions over repetition for building nested lists.
- Use `id()` and `is` only when reasoning about identity, not value equality.

## Optional extension
Try repeating these exercises with `set`, `bytearray`, custom classes, and NumPy arrays to see how mutability interacts with methods, slicing, and views.
